In [3]:
## import libraries
import tskit
import numpy as np
import pandas as pd
import os
from typing import Dict, Any, List
from scipy.spatial import distance_matrix
from collections import defaultdict


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import tskit
from bisect import bisect_right

# Updated input pattern
#input_pattern = "/lisc/data/scratch/admixlab/hackl/treesequences_subsetAfricansphased/tsinfer_results/subset_chr{chrom}_tsinfer.ts"
input_pattern = "/lisc/data/scratch/admixlab/hackl/treesequences_subsetAfricansphased_rec/tsinfer_results/subset_chr{chrom}_tsinfer.ts"
output_file = "/lisc/data/scratch/admixlab/mk_data/san/all_chroms_subset.trees"
bed_file = "/lisc/data/scratch/admixlab/mk_data/san/bestregions.bed"

def merge_intervals(intervals):
    # Merge overlapping or touching intervals [start, end)
    if not intervals:
        return []
    ivals = sorted((int(s), int(e)) for s, e in intervals if int(e) > int(s))
    merged = [list(ivals[0])]
    for s, e in ivals[1:]:
        if s <= merged[-1][1]:  # overlap or touching
            if e > merged[-1][1]:
                merged[-1][1] = e
        else:
            merged.append([s, e])
    return [(s, e) for s, e in merged]

def read_bed_by_chrom(bed_path):
    # Read BED (0-based, half-open) and return dict chrom -> merged intervals
    df = pd.read_csv(bed_path, sep="\t", header=None, usecols=[0, 1, 2],
                     names=["chrom", "start", "end"])
    df["start"] = df["start"].astype(np.int64)
    df["end"] = df["end"].astype(np.int64)
    by_chrom = {}
    for chrom, sub in df.groupby("chrom"):
        intervals = list(zip(sub["start"].tolist(), sub["end"].tolist()))
        by_chrom[chrom] = merge_intervals(intervals)
    return by_chrom

def prefix_lengths(intervals):
    starts = [s for s, _ in intervals]
    lens = [e - s for s, e in intervals]
    comp_offsets = np.zeros(len(intervals), dtype=np.int64)
    if len(intervals) > 1:
        comp_offsets[1:] = np.cumsum(lens[:-1], dtype=np.int64)
    return starts, comp_offsets, lens

def find_interval_index(starts, x):
    j = bisect_right(starts, x) - 1
    return j

def assert_integral_positions(ts, chrom_label):
    pos = ts.tables.sites.position
    if len(pos) == 0:
        return
    diffs = np.abs(pos - np.round(pos))
    if np.any(diffs > 1e-9):
        bad = np.where(diffs > 1e-9)[0][:5].tolist()
        raise ValueError(f"Non-integral site positions found in {chrom_label}; example indices: {bad}")

def main():
    bed_by_chrom = read_bed_by_chrom(bed_file)
    chrom_labels = [f"chr{i}" for i in range(1, 23)]

    # Bootstrap from chromosome 1
    first_chrom = 1
    first_label = f"chr{first_chrom}"
    if first_label not in bed_by_chrom:
        raise ValueError(f"No BED intervals for {first_label}")
    first_ts_path = input_pattern.format(chrom=first_chrom)
    if not os.path.exists(first_ts_path):
        raise FileNotFoundError(first_ts_path)
    ts0 = tskit.load(first_ts_path)
    assert_integral_positions(ts0, first_label)

    # Compute preserved lengths and offsets
    per_chrom_pres_len = {lab: int(sum(e - s for s, e in bed_by_chrom.get(lab, [])))
                          for lab in chrom_labels}
    chrom_offsets = {}
    running = 0
    for lab in chrom_labels:
        chrom_offsets[lab] = running
        running += per_chrom_pres_len[lab]
    total_preserved_len = running

    # New tables
    new_tables = tskit.TableCollection(sequence_length=float(total_preserved_len))

    # Copy schemas
    try:
        new_tables.metadata_schema = ts0.tables.metadata_schema
    except Exception:
        pass  # older tskit may not support collection-level metadata schema
    new_tables.populations.metadata_schema = ts0.tables.populations.metadata_schema
    new_tables.individuals.metadata_schema = ts0.tables.individuals.metadata_schema
    new_tables.nodes.metadata_schema = ts0.tables.nodes.metadata_schema
    new_tables.edges.metadata_schema = ts0.tables.edges.metadata_schema
    new_tables.sites.metadata_schema = ts0.tables.sites.metadata_schema
    new_tables.mutations.metadata_schema = ts0.tables.mutations.metadata_schema
    new_tables.migrations.metadata_schema = ts0.tables.migrations.metadata_schema

    # Copy populations, individuals, and nodes from chr1 by setting columns
    new_tables.populations.set_columns(**ts0.tables.populations.asdict())
    new_tables.individuals.set_columns(**ts0.tables.individuals.asdict())
    new_tables.nodes.set_columns(**ts0.tables.nodes.asdict())

    # Global sample node order (from first ts)
    global_samples = ts0.samples().tolist()
    n_global_samples = len(global_samples)

    # Per-chrom local->global node map cache
    node_map_cache = {}
    node_map_cache[first_label] = np.arange(ts0.num_nodes, dtype=np.int64)

    def build_node_map_for_ts(ts, lab):
        if lab in node_map_cache:
            return node_map_cache[lab]
        local_to_global = np.full(ts.num_nodes, -1, dtype=np.int64)
        # Map samples by order
        local_samples = ts.samples()
        if len(local_samples) != n_global_samples:
            raise ValueError(f"Sample count mismatch in {lab}: {len(local_samples)} vs {n_global_samples}")
        for rank, loc_id in enumerate(local_samples):
            local_to_global[loc_id] = global_samples[rank]
        # Append internal nodes
        nt = ts.tables.nodes
        for loc_id in range(ts.num_nodes):
            if local_to_global[loc_id] != -1:
                continue
            row = nt[loc_id]
            gid = new_tables.nodes.add_row(
                flags=row.flags,
                time=row.time,
                population=row.population,
                individual=row.individual,
                metadata=row.metadata,
            )
            local_to_global[loc_id] = gid
        node_map_cache[lab] = local_to_global
        return local_to_global

    def append_chrom(ts, lab, intervals):
        assert_integral_positions(ts, lab)
        L = int(ts.sequence_length)
        for s, e in intervals:
            if not (0 <= s < e <= L):
                raise ValueError(f"BED interval out of bounds for {lab}: {(s, e)} vs L={L}")
        if len(intervals) == 0:
            return

        kept_ts = ts.keep_intervals(intervals, simplify=False)
        starts, comp_offsets, lengths = prefix_lengths(intervals)
        assert sum(lengths) == per_chrom_pres_len[lab], "Per-chrom preserved length mismatch"
        node_map = build_node_map_for_ts(ts, lab)
        g_off = chrom_offsets[lab]

        # Edges
        for e in kept_ts.tables.edges:
            j = find_interval_index(starts, int(e.left))
            s_j = starts[j]
            comp_j = comp_offsets[j]
            new_left = g_off + comp_j + (int(e.left) - s_j)
            new_right = g_off + comp_j + (int(e.right) - s_j)
            if not (new_left < new_right):
                continue
            new_parent = int(node_map[e.parent])
            new_child = int(node_map[e.child])
            new_tables.edges.add_row(
                left=float(new_left),
                right=float(new_right),
                parent=new_parent,
                child=new_child,
                metadata=e.metadata,
            )

        # Sites: map local site index -> new site id
        site_id_map = {}
        for s_idx, site in enumerate(kept_ts.tables.sites):
            j = find_interval_index(starts, int(site.position))
            s_j = starts[j]
            comp_j = comp_offsets[j]
            new_pos = g_off + comp_j + (int(site.position) - s_j)
            new_site_id = new_tables.sites.add_row(
                position=float(new_pos),
                ancestral_state=site.ancestral_state,
                metadata=site.metadata,
            )
            site_id_map[s_idx] = new_site_id

        # Mutations: preserve order, remap site/node/parent.
        # Use row indices for mapping (not mut.id).
        old_to_new_mut_id = {}
        for mut_idx, mut in enumerate(kept_ts.tables.mutations):
            # Enforce parent-before-child in the kept mutation table
            if mut.parent != tskit.NULL and mut.parent not in old_to_new_mut_id:
                raise ValueError(
                    f"Unexpected mutation ordering in {lab}: parent row {mut.parent} "
                    f"not seen before child at row {mut_idx}"
                )
            new_site_id = site_id_map[mut.site]
            new_node_id = int(node_map[mut.node])
            new_parent_mut = tskit.NULL if mut.parent == tskit.NULL else old_to_new_mut_id[mut.parent]
            new_mut_id = new_tables.mutations.add_row(
                site=new_site_id,
                node=new_node_id,
                derived_state=mut.derived_state,
                parent=new_parent_mut,
                time=mut.time,
                metadata=mut.metadata,
            )
            old_to_new_mut_id[mut_idx] = new_mut_id

    # Append each chromosome (including chr1 for edges/sites/mutations)
    for i in range(1, 23):
        lab = f"chr{i}"
        ts_path = input_pattern.format(chrom=i)
        if not os.path.exists(ts_path):
            raise FileNotFoundError(ts_path)
        ts = tskit.load(ts_path)

        # Basic consistency checks with the first ts
        if ts.tables.populations.num_rows != ts0.tables.populations.num_rows:
            raise ValueError(f"Population table size differs in {lab}")
        if ts.tables.individuals.num_rows != ts0.tables.individuals.num_rows:
            raise ValueError(f"Individual table size differs in {lab}")

        ivals = bed_by_chrom.get(lab, [])
        if per_chrom_pres_len[lab] != sum((e - s) for s, e in ivals):
            raise AssertionError("Internal preserved length accounting mismatch")

        append_chrom(ts, lab, ivals)

    # Finalize
    new_tables.sequence_length = float(total_preserved_len)
    new_tables.sort()

    # Try to add a provenance record in a version-tolerant way
    prov_record = {
        "software": "custom-tskit-concat",
        "version": "1.0",
        "inputs": {
            "input_pattern": input_pattern,
            "bed_file": bed_file,
            "chrom_labels": chrom_labels,
        }
    }
    try:
        # Newer tskit expects JSON string
        new_tables.provenances.add_row(record=json.dumps(prov_record))
    except TypeError:
        try:
            new_tables.provenances.add_row(tskit.PROVENANCE_SCHEMA.validate_and_encode_row(prov_record))
        except Exception:
            pass

    new_ts = new_tables.tree_sequence()
    assert int(new_ts.sequence_length) == total_preserved_len, (
        f"Final length {new_ts.sequence_length} != {total_preserved_len}"
    )
    new_ts.dump(output_file)
    print(f"Written: {output_file}")
    print(f"Total preserved length: {total_preserved_len}")
    assert list(new_ts.samples()) == list(ts0.samples())
    print("Sample nodes preserved and identical.")

if __name__ == "__main__":
    main()

In [ ]:
ts2 = tskit.load('/lisc/data/scratch/admixlab/mk_data/test/test.ts')

ts2

In [21]:
## modify metadata of tree sequence to match slendr expectations, and save final real data file!
ts = tskit.load("/lisc/data/scratch/admixlab/mk_data/san/all_chroms_subset.trees")

# Read the sample information from file
with open("/lisc/home/user/kuhlwilm/demog-afr/preprocess/samplenames.txt", "r") as f:
    # Read all lines and extract sample names and populations
    lines = [line.strip().split() for line in f if line.strip()]
    new_sample_names = [line[0] for line in lines]
    new_population_names = [line[1] for line in lines]

# Create new tables
tables = ts.dump_tables()

# Step 1: Update ts.populations table with new population names
# Clear and rebuild the populations table with new population names
tables.populations.clear()

# Create a mapping from population name to ID
pop_name_to_id = {}
for i, pop_name in enumerate(sorted(set(new_population_names))):
    # Add each unique population to the table
    tables.populations.add_row(metadata={"name": pop_name})
    pop_name_to_id[pop_name] = i  # Store the ID for later use

# Step 2: Update ts.individuals table
# First, get all the node assignments
individual_nodes = [ind.nodes for ind in ts.individuals()]

# Clear the individuals table
tables.individuals.clear()

# Create a list to store the new individual IDs
new_individual_ids = []

# Add individuals with updated information
for i in range(ts.num_individuals):
    # Get the new sample name and population name for this individual
    sample_name = new_sample_names[i]
    population_name = new_population_names[i]

    # Get the population ID for this individual
    population_id = pop_name_to_id[population_name]

    # Create new metadata with only the required fields
    metadata = {
        "name": sample_name,
        "population": population_name,
        "population_id": int(population_id)  # This must match the actual population ID
    }

    # Add the individual with updated information
    ind_id = tables.individuals.add_row(
        flags=1,  # Set flags to 1 for all individuals
        location=ts.individual(i).location,  # Keep original location
        metadata=metadata
    )
    new_individual_ids.append(ind_id)

# Step 3: Update the node table to reference the new individuals
# and set their population IDs correctly
node_table = tables.nodes
for i, nodes in enumerate(individual_nodes):
    for node_id in nodes:
        # Get the current node
        node = node_table[node_id]
        # Update the individual reference and population ID
        population_name = new_population_names[i]
        population_id = pop_name_to_id[population_name]
        node_table[node_id] = node.replace(
            individual=new_individual_ids[i],
            population=population_id
        )

# Rebuild the TreeSequence
ts_updated = tables.tree_sequence()

# Verify the changes
print("Updated populations:")
for pop in ts_updated.populations():
    print(f"Population {pop.id}: {pop.metadata}")

print("\nUpdated individuals (first 5):")
for i, ind in enumerate(ts_updated.individuals()):
    if i < 5:  # Print first 5 individuals for verification
        print(f"Individual {ind.id}:")
        print(f"  Flags: {ind.flags}")
        print(f"  Nodes: {ind.nodes}")
        print(f"  Metadata: {ind.metadata}")
        print(f"  Population ID: {ind.metadata['population_id']}")
        print()

# Verify node population assignments
print("\nNode population assignments (first 10 nodes):")
for i, node in enumerate(ts_updated.nodes()):
    if i < 10:  # Print first 10 nodes for verification
        print(f"Node {node.id}: Population {node.population}")

# Rebuild the TreeSequence
ts5 = ts_updated

# Save the updated TreeSequence
ts5.dump('/lisc/data/scratch/admixlab/mk_data/san/all_chroms.trees')    


Updated populations:
Population 0: {'name': 'KS'}
Population 1: {'name': 'RHGn'}
Population 2: {'name': 'eRHG'}
Population 3: {'name': 'wRHG'}

Updated individuals (first 5):
Individual 0:
  Flags: 1
  Nodes: [0 1]
  Metadata: {'variant_data_time': 0, 'name': 'KS_1', 'population': 'KS', 'population_id': 0}
  Population ID: 0

Individual 1:
  Flags: 1
  Nodes: [2 3]
  Metadata: {'variant_data_time': 0, 'name': 'KS_2', 'population': 'KS', 'population_id': 0}
  Population ID: 0

Individual 2:
  Flags: 1
  Nodes: [4 5]
  Metadata: {'variant_data_time': 0, 'name': 'KS_3', 'population': 'KS', 'population_id': 0}
  Population ID: 0

Individual 3:
  Flags: 1
  Nodes: [6 7]
  Metadata: {'variant_data_time': 0, 'name': 'KS_4', 'population': 'KS', 'population_id': 0}
  Population ID: 0

Individual 4:
  Flags: 1
  Nodes: [8 9]
  Metadata: {'variant_data_time': 0, 'name': 'KS_5', 'population': 'KS', 'population_id': 0}
  Population ID: 0


Node population assignments (first 10 nodes):
Node 0: Popul

In [22]:
import tskit
import numpy as np

final_path = "/lisc/data/scratch/admixlab/mk_data/san/all_chroms.trees"

def as_text(x):
    # tskit stores states as bytes or strings depending on version/source.
    if x is None:
        return None
    if isinstance(x, bytes):
        try:
            return x.decode("utf-8")
        except Exception:
            # Fallback to repr for non-UTF8
            return repr(x)
    return str(x)

def main():
    ts = tskit.load(final_path)
    tables = ts.tables
    muts = tables.mutations
    sites = tables.sites

    n = muts.num_rows
    if n == 0:
        print("No mutations in the tree sequence.")
        return

    start = max(0, n - 20)
    print(f"Total mutations: {n}. Printing rows [{start}..{n-1}]")

    # Convenience views
    site_pos = sites.position  # numpy array of float64
    # We’ll format positions as integers if they’re integral.
    def fmt_pos(p):
        if abs(p - round(p)) < 1e-9:
            return str(int(round(p)))
        return f"{p:.6f}"

    # Some versions expose arrays for derived_state/ancestral_state only via iteration; we pull per-row
    for i in range(start, n):
        m = muts[i]
        s = sites[m.site]
        pos_str = fmt_pos(site_pos[m.site])

        # Derived state text
        dstate = as_text(m.derived_state)
        astate = as_text(s.ancestral_state)

        # Node/sample flag info (optional)
        node_flags = tables.nodes[m.node].flags
        is_sample = bool(node_flags & tskit.NODE_IS_SAMPLE)

        print(
            f"mut_row={i} site={m.site} pos={pos_str} "
            f"node={m.node}{' (sample)' if is_sample else ''} "
            f"parent={m.parent} time={m.time} "
            f"anc={astate} der={dstate}"
        )

if __name__ == "__main__":
    main()

Total mutations: 3946086. Printing rows [3946066..3946085]
mut_row=3946066 site=3443460 pos=249997747 node=196 (sample) parent=-1 time=1532.527439656899 anc=T der=C
mut_row=3946067 site=3443461 pos=249997786 node=38 (sample) parent=-1 time=2458.068570170354 anc=C der=G
mut_row=3946068 site=3443463 pos=249998006 node=17382093 parent=-1 time=3773.848433388777 anc=T der=G
mut_row=3946069 site=3443463 pos=249998006 node=105 (sample) parent=-1 time=2357.38466017899 anc=T der=G
mut_row=3946070 site=3443464 pos=249998021 node=194 (sample) parent=-1 time=1561.637992055945 anc=G der=A
mut_row=3946071 site=3443465 pos=249998046 node=17435781 parent=-1 time=5904.383227393147 anc=C der=T
mut_row=3946072 site=3443465 pos=249998046 node=189 (sample) parent=3946071 time=1864.9232283103481 anc=C der=C
mut_row=3946073 site=3443466 pos=249998147 node=197 (sample) parent=-1 time=4051.7928155001855 anc=C der=T
mut_row=3946074 site=3443467 pos=249998184 node=140 (sample) parent=-1 time=1558.391918908924 an